# Memory-Split PoC — Optimal (GPT-5.6-sol) Retriever

**Question.** At a fixed parameter budget, does a model that *offloads facts* to an external retriever (**SPLIT**) beat a dense twin that stores them in-weights (**DENSE**) once retrieval is perfect?

We assume an **optimal retriever** whose golden knowledge is generated by **GPT-5.6-sol** (`openai-group/gpt-5.6-sol`) via the TrueFoundry gateway. Facts are **real Wikidata triples** (PopQA), so GPT genuinely knows them. Single-hop; the two arms share one corpus, model, budget, and init — the only difference is whether fact values carry training loss.

**Three eval conditions:** DENSE @ closed-book, DENSE + oracle (RAG: fact in context), SPLIT @ GPT-oracle (+ a SPLIT @ gold upper bound).

**Scale.** Defaults to the team's primary scale **d160m (~162M params)**; `--model d360m` (~356M) for the stretch (needs A100).

**Runtime:** pick a **GPU** runtime (A100 recommended). Run cells top to bottom. You'll paste a TrueFoundry token when prompted.

**Caveats:** the held-out fact-QA gap is partly definitional; the seen split is the fairer capacity comparison; the token budget is far smaller than the team's cluster runs, so the reasoning composite is indicative, not final.

## 1. Get the code (always syncs to the latest branch state)

In [ ]:
import os
REPO_URL = "https://github.com/sidvenkatayogi/Memory-Split.git"
BRANCH = "poc/optimal-retriever"
if os.path.basename(os.getcwd()) != "Memory-Split":
    if not os.path.isdir("Memory-Split"):
        !git clone --branch {BRANCH} --single-branch {REPO_URL}
    %cd Memory-Split
# Sync to the latest code on the branch so a re-run picks up new evals
# (e.g. the DENSE + oracle RAG arm) instead of a stale cached clone.
# Uses FETCH_HEAD to avoid the slash in the branch name; data/ is gitignored
# so your local corpus/checkpoints are untouched.
!git fetch -q origin {BRANCH} && git reset --hard -q FETCH_HEAD
!git log --oneline -1

In [ ]:
# Colab preinstalls torch/numpy/matplotlib/pyyaml with a CUDA-matched build.
# Install ONLY the missing runtime deps so we never reinstall (and risk
# breaking GPU) torch. `datasets` is not needed at runtime (facts are committed).
!pip install -q tiktoken openai
import torch; print('torch', torch.__version__, '| cuda:', torch.cuda.is_available(),
                    '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

## 2. Persist to Google Drive (checkpoints + golden cache + results)
Points `POC_PERSIST_DIR` at a Drive folder. Checkpoints, the (paid) GPT golden cache, and results live there and **survive a disconnect** — so a re-run reuses the already-trained models instead of retraining, and re-uses cached GPT answers instead of re-calling. The corpus stays local and is rebuilt byte-identically from the same seeds.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
os.environ['POC_PERSIST_DIR'] = '/content/drive/MyDrive/memory_split_poc'
os.makedirs(os.environ['POC_PERSIST_DIR'], exist_ok=True)
print('persist dir ->', os.environ['POC_PERSIST_DIR'])

## 3. TrueFoundry credentials
Paste your TrueFoundry token (used as `OPENAI_API_KEY`). The smoke call fails fast here if the token/model is wrong.

In [ ]:
import os, getpass
os.environ["OPENAI_API_KEY"] = getpass.getpass("TrueFoundry token: ")
os.environ["OPENAI_BASE_URL"] = "https://tfy.promptlens.trilogy.com/v1"
os.environ["POC_GPT_MODEL"] = "openai-group/gpt-5.6-sol"

import sys; sys.path.insert(0, ".")
from evals.gpt_oracle import GatewayClient
print("gateway smoke ->", GatewayClient().smoke())  # e.g. 'Paris'

## 4. Build the shared corpus (offline; fast; regenerates the local corpus + eval sets)

In [ ]:
!python scripts/poc_run.py --stage build

## 5. Train the matched twins (DENSE then SPLIT)
**Reuses existing checkpoints:** if a finished checkpoint for this `--model`/`--steps` is already on Drive, training is **skipped** and the trained model is reused (no retraining). Pass `--fresh` to force retrain, or raise `--steps` to continue training.

In [ ]:
!python scripts/poc_run.py --stage train --device auto --model d160m --steps 4000 --ckpt-minutes 5

## 6. Generate golden knowledge with GPT-5.6-sol (cached; already-generated items are not re-called)

In [ ]:
!python scripts/poc_run.py --stage gen-golden

## 7. Evaluate + report
Three fact-QA conditions (DENSE closed-book, DENSE + oracle RAG, SPLIT @ GPT-oracle) + SPLIT @ gold upper bound, plus the reasoning composite. Loads the trained models from Drive.

In [ ]:
!python scripts/poc_run.py --stage eval --gold-oracle --device auto
!python scripts/poc_run.py --stage report

In [ ]:
import json, os
from IPython.display import Image, display
persist = os.environ.get('POC_PERSIST_DIR', 'data/poc')
print(json.dumps(json.load(open(f'{persist}/poc_results.json')), indent=2))
display(Image(f'{persist}/poc_figure.png'))